# PIPELINE HUẤN LUYỆN S2ST ANH - VIỆT TRÊN MOLAB (RTX 6000 & MARIMO)

Notebook này được đồng bộ 1:1 với file script `molab_marimo_training_pipeline.py`.
Bạn có thể:
1. **Chạy trực tiếp tại đây trên Jupyter / VS Code**.
2. **Hoặc mở bằng Marimo Web UI**: `marimo edit molab_marimo_training_pipeline.py`.

### Các tính năng cốt lõi đã tích hợp:
- **ToneAwareBridge**: Conv1D downsample 50Hz -> 25Hz + Transformer 6 layers + Phân loại 6 thanh điệu tiếng Việt.
- **FlowMatchingTargetDecoder**: Conditional Flow Matching (CFM) giải Euler ODE 25 bước sinh Mel-spectrogram.
- **Streaming Dataloader**: Nạp Google FLEURS (en_us -> vi_vn) không tốn dung lượng ổ cứng (có fallback dummy offline).
- **Mixed Precision (bfloat16)**: Tận dụng tối đa tốc độ của GPU NVIDIA RTX 6000 trên MoLab.
- **Tự động lưu Checkpoint**: Xuất các file `.pt` sau mỗi epoch để phục vụ đo lường và inference.

In [ ]:
# Cell 1: Khởi động huấn luyện trực tiếp
from molab_marimo_training_pipeline import run_training

# Cấu hình tham số huấn luyện:
# Khi test thử: epochs=3, steps_per_epoch=10, batch_size=4
# Khi train thật trên MoLab: epochs=30, steps_per_epoch=100, batch_size=16 hoặc 32
model = run_training(
    epochs=3,
    steps_per_epoch=10,
    batch_size=8,
    lr=2e-4,
    checkpoint_dir="checkpoints_molab",
    encoder_dim=1280, # Chuẩn Whisper-large-v3
    bridge_dim=512,
    bridge_layers=6
)


In [ ]:
# Cell 2: Chạy suy luận (Zero-shot Inference) câu mới
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Giả lập một câu tiếng Anh mới từ Whisper Encoder [1, 100 frames, 1280]
sample_whisper = torch.randn(1, 100, 1280, device=device)
# Vector giọng nói 192 chiều từ ECAPA-TDNN của người nói tiếng Anh
sample_spk = torch.randn(1, 192, device=device)

# Sinh Mel-spectrogram tiếng Việt
generated_mel = model.inference(sample_whisper, sample_spk, num_steps=25)

print("Shape Mel-spectrogram tiếng Việt:", generated_mel.shape)
print("-> Mel này sẵn sàng đưa vào Vocos / HiFi-GAN Vocoder để xuất ra file .wav!")
